# 02 — DPO Alignment

This notebook fine-tunes the SFT model using **Direct Preference Optimization (DPO)** on the `(prompt, chosen, rejected)` triples from our annotation dataset.

DPO vs PPO: PPO requires training a separate reward model and running RL with it in the loop. DPO reformulates preference learning as a classification problem directly on the policy — no reward model needed, no RL instability. Given 214 preference pairs and a small model, DPO is both more stable and more compute-efficient.

In [ ]:
import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import subprocess, sys
repo_root = "/content/aipi590-challenge-2"
token = os.environ["GITHUB_TOKEN"]
if not os.path.exists(repo_root):
    subprocess.run(
        f"git clone https://{token}@github.com/jonasneves/aipi590-challenge-2.git {repo_root}",
        shell=True, check=True
    )

!pip install -q trl peft transformers datasets accelerate bitsandbytes

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Always reload colab_utils so a re-run picks up the latest version from the repo
import importlib, src.colab_utils as _cu
importlib.reload(_cu)
from src.colab_utils import prepare_notebook, ensure_requirements, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")

# Reload again after prepare_notebook pulls the latest code
importlib.reload(_cu)
from src.colab_utils import prepare_notebook, ensure_requirements, publish_artifacts

ensure_requirements(repo_root)


In [ ]:
from src.dataset import load_dpo_dataset

pref_path = paths["data"] / "preferences.json"
dataset = load_dpo_dataset(pref_path)
print(f"Train: {len(dataset['train'])} examples")
print(f"Val:   {len(dataset['val'])} examples")
print("\nSample triple:")
sample = dataset["train"][0]
print(f"  prompt:   {sample['prompt']}")
print(f"  chosen:   {sample['chosen']}")
print(f"  rejected: {sample['rejected']}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

SFT_PATH = "/content/sft_model"
DRIVE_SFT = "/content/drive/MyDrive/aipi590-challenge-2/sft_model"
MODEL_NAME = "distilgpt2"

if Path(SFT_PATH).exists():
    model_path = SFT_PATH
    print(f"Loading SFT model from {SFT_PATH}")
elif Path(DRIVE_SFT).exists():
    model_path = DRIVE_SFT
    print(f"Loading SFT model from Drive: {DRIVE_SFT}")
else:
    model_path = MODEL_NAME
    print("WARNING: SFT model not found in /content or Drive -- falling back to base distilgpt2")

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_path)

# π_ref = frozen SFT model (Lecture 7: choose a strong π_ref)
ref_model = AutoModelForCausalLM.from_pretrained(model_path)
for p in ref_model.parameters():
    p.requires_grad_(False)
print(f"Reference model loaded and frozen from: {model_path}")


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="/content/dpo_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-5,   # lowered from 5e-5 for stability
    beta=0.05,             # lowered from 0.1 — Lecture 7: calibrate β on held-out win-rate
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    max_length=256,
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,   # π_ref = frozen SFT model (Lecture 7)
    args=dpo_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    processing_class=tokenizer,
)
trainer.train()

# Persist log history
import json as _json
(paths["results"] / "dpo_log_history.json").write_text(
    _json.dumps(trainer.state.log_history, indent=2)
)


In [ ]:
import shutil
from pathlib import Path

merged = model.merge_and_unload()
merged.save_pretrained("/content/dpo_model")
tokenizer.save_pretrained("/content/dpo_model")
print("DPO model saved to /content/dpo_model")

drive_path = Path("/content/drive/MyDrive/aipi590-challenge-2/dpo_model")
drive_path.mkdir(parents=True, exist_ok=True)
shutil.copytree("/content/dpo_model", drive_path, dirs_exist_ok=True)
print(f"DPO model backed up to {drive_path}")


In [ ]:
import matplotlib.pyplot as plt
import json

log_history = json.loads((paths["results"] / "dpo_log_history.json").read_text())

loss_entries     = [(e["step"], e["loss"])             for e in log_history if "loss" in e]
chosen_entries   = [(e["step"], e["rewards/chosen"])   for e in log_history if "rewards/chosen" in e]
rejected_entries = [(e["step"], e["rewards/rejected"]) for e in log_history if "rewards/rejected" in e]

n_plots = 1 + (1 if chosen_entries else 0)
fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

if loss_entries:
    steps, losses = zip(*loss_entries)
    axes[0].plot(steps, losses)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("DPO Training Loss")

if chosen_entries and len(axes) > 1:
    c_steps, c_vals = zip(*chosen_entries)
    r_steps, r_vals = zip(*rejected_entries)
    axes[1].plot(c_steps, c_vals, label="chosen")
    axes[1].plot(r_steps, r_vals, label="rejected")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Reward")
    axes[1].set_title("Rewards: chosen vs rejected")
    axes[1].legend()

plt.tight_layout()
chart_path = paths["results"] / "dpo_loss.png"
plt.savefig(chart_path, dpi=150)
plt.show()

metrics = {
    "final_loss":             loss_entries[-1][1]     if loss_entries     else None,
    "steps":                  len(loss_entries),
    "final_reward_chosen":    chosen_entries[-1][1]   if chosen_entries   else None,
    "final_reward_rejected":  rejected_entries[-1][1] if rejected_entries else None,
}
metrics_path = paths["results"] / "dpo_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))

publish_artifacts([chart_path, metrics_path], "add dpo training results", repo_root)
